# Real-ESRGAN 视频增强（Kaggle T4 × 2）

目标是 **1920×1080 → 3840×2160**。上次测试参数 `SCALE=4` 实际生成了 `7680×4320`（8K），因此 852.4 秒和 38.3 MiB 不能作为 2160p 基准。本版恢复 `SCALE=2`，优先使用双卡整帧交错推理；显存不足时再回退到重叠 tile。请启用 **GPU T4 x2** 和 Internet。

In [ ]:
!git clone --depth 1 https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN

In [ ]:
!cd /kaggle/working/Real-ESRGAN
!pip install -q -r requirements-video-kaggle.txt
!pip install -q -e . --no-deps

In [ ]:
# 环境、编码器和 NVENC 运行时检查
import cv2
import json
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("opencv:", cv2.__version__)
print("CUDA devices:", torch.cuda.device_count())
for gpu_id in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_id)
    print(gpu_id, props.name, f"{props.total_memory / 1024**3:.1f} GiB")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "请在 Kaggle 设置中选择 GPU T4 x2"
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["ffprobe", "-version"], check=True, stdout=subprocess.DEVNULL)
encoder_probe = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], check=True, capture_output=True, text=True)
encoder_text = encoder_probe.stdout + encoder_probe.stderr
assert "libx264" in encoder_text, "当前 ffmpeg 缺少 libx264 编码器"
NVENC_OK = False
if "hevc_nvenc" in encoder_text:
    smoke = subprocess.run(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", "color=size=128x128:rate=1", "-frames:v", "1", "-c:v", "hevc_nvenc", "-gpu", "0", "-preset", "p7", "-tune", "hq", "-rc", "vbr", "-cq", "18", "-b:v", "0", "-multipass", "fullres", "-spatial_aq", "1", "-temporal_aq", "1", "-rc-lookahead", "32", "-bf", "3", "-pix_fmt", "yuv420p", "-f", "null", "-"], capture_output=True, text=True)
    NVENC_OK = smoke.returncode == 0
    if not NVENC_OK:
        print("hevc_nvenc smoke test 失败，将回退 libx264:", smoke.stderr[-500:])
print("ffmpeg/ffprobe/libx264: OK; hevc_nvenc runtime:", NVENC_OK)

## 参数

`TILE_SIZE=0` 表示整帧推理：两张 GPU 同时处理相邻帧，无 tile 重叠计算和接缝，通常也是最高画质路径。1080p + animevideov3 + FP16 在 T4 上应优先测试此模式。若 OOM，按文末方案回退。`hevc_nvenc` 使用 T4 独立编码单元，在高质量设置下通常比 x264 更快且更小；若运行时不可用会自动选择 libx264。

In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/1.mkv"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"
MODEL = "realesr-animevideov3"
MODEL_PATH = ""
DENOISE_STRENGTH = 1.0
SCALE = 2                       # 1080p → 2160p；4.0 会生成 8K
INPUT_WIDTH = 0
INPUT_HEIGHT = 0
TILE_SIZE = 0                     # 最快整帧模式；OOM 时见文末回退方案
OVERLAP = 0
BATCH_SIZE = 8                    # tile 模式每卡批量；整帧模式固定每卡一帧
GPU_IDS = "0,1"
VIDEO_CODEC = "hevc_nvenc" if NVENC_OK else "libx264"
CRF = 18                          # libx264/libx265
PRESET = "medium"
CQ = 18                           # NVENC；越低质量越高
NVENC_PRESET = "p7"            # 质量优先；p6 更快
ENCODE_GPU = 0
AUDIO_CODEC = "aac"            # 10秒测试精确同步；完整视频可用 copy
AUDIO_BITRATE = "192k"
START_TIME = 60 * 8
TEST_SECONDS = 10.0               # 改为 0 处理到视频末尾
PROGRESS_INTERVAL = 60.0

assert Path(INPUT_VIDEO).is_file(), f"输入不存在，请修改 INPUT_VIDEO: {INPUT_VIDEO}"
print("编码器:", VIDEO_CODEC, "目标输出: 3840x2160")

In [ ]:
!python inference_realesrgan_video_kaggle.py \
  --input "{INPUT_VIDEO}" \
  --output "{OUTPUT_VIDEO}" \
  --model "{MODEL}" \
  --model-path "{MODEL_PATH}" \
  --denoise-strength {DENOISE_STRENGTH} \
  --scale {SCALE} \
  --fp16 \
  --channels-last \
  --input-width {INPUT_WIDTH} \
  --input-height {INPUT_HEIGHT} \
  --tile-size {TILE_SIZE} \
  --overlap {OVERLAP} \
  --batch-size {BATCH_SIZE} \
  --gpu-ids "{GPU_IDS}" \
  --video-codec "{VIDEO_CODEC}" \
  --crf {CRF} \
  --preset "{PRESET}" \
  --cq {CQ} \
  --nvenc-preset "{NVENC_PRESET}" \
  --encode-gpu {ENCODE_GPU} \
  --audio-codec "{AUDIO_CODEC}" \
  --audio-bitrate "{AUDIO_BITRATE}" \
  --start-time {START_TIME} \
  --test-seconds {TEST_SECONDS} \
  --progress-interval {PROGRESS_INTERVAL} \
  --ffmpeg-bin ffmpeg \
  --ffprobe-bin ffprobe

In [ ]:
# 必须验证分辨率、帧率、时长、音频、体积和平均码率，避免再次把 8K 当 2160p。
assert Path(OUTPUT_VIDEO).is_file(), "输出视频未生成，请查看上一个单元格日志"
probe = subprocess.run(["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", OUTPUT_VIDEO], check=True, capture_output=True, text=True)
metadata = json.loads(probe.stdout)
video_stream = next(s for s in metadata["streams"] if s["codec_type"] == "video")
audio_streams = [s for s in metadata["streams"] if s["codec_type"] == "audio"]
size_mib = Path(OUTPUT_VIDEO).stat().st_size / 1024**2
duration = float(metadata["format"]["duration"])
bitrate_mbps = Path(OUTPUT_VIDEO).stat().st_size * 8 / duration / 1_000_000
summary = {"codec": video_stream["codec_name"], "size": f'{video_stream["width"]}x{video_stream["height"]}', "pix_fmt": video_stream.get("pix_fmt"), "fps": video_stream.get("avg_frame_rate"), "duration_s": duration, "audio_codec": audio_streams[0]["codec_name"] if audio_streams else None, "file_MiB": round(size_mib, 2), "bitrate_Mbps": round(bitrate_mbps, 2)}
print(json.dumps(summary, ensure_ascii=False, indent=2))